In [ ]:
import yaml
import os
from pathlib import Path
import jax.numpy as jnp
import matplotlib.pyplot as plt
from generative_uncertainty.compute_uncertainty import get_uncertainty_scores 
from generative_uncertainty.plots import plot_uncertainty_threshold_analysis
from generative_uncertainty.config import load_config
from generative_uncertainty.plots import show_scatter_with_threshold
from generative_uncertainty.plots import show_hallucination_analysis
import os
%matplotlib inline
%load_ext autoreload
%autoreload 2

config = "config.yml"
config_data = {}
if os.path.exists(config):
    with open(config, 'r') as f:
        config_data = load_config(config) 
else:
    print(f"Warning: Config file '{config}' not found. Using defaults.")

trained_models_dir = config_data.deep_ensemble.trained_models_dir
real_dataset_path = trained_models_dir.format(seed=0) + "/real_dataset.npy"
real_data = jnp.load(real_dataset_path)
print(f"loaded real data : {real_data.shape}")

samples_cache_dir = config_data.sampling.samples_cache_dir
from shared_utils.colors import okabe_ito as colors

In [ ]:
ensemble_samples_deep = jnp.load(Path(samples_cache_dir) / "deep_ensemble_samples.npy")
ensemble_samples_lora = jnp.load(Path(samples_cache_dir) / "lora_ensemble_samples.npy")
# print(f"loaded ensemble samples: {ensemble_samples_la.shape}")
base_samples_deep = ensemble_samples_deep[0]
base_samples_lora = ensemble_samples_lora[0]

uncertainty_scores_deep = get_uncertainty_scores(ensemble_samples_deep)
uncertainty_scores_lora = get_uncertainty_scores(ensemble_samples_lora)


In [ ]:
plot_uncertainty_threshold_analysis(uncertainty_scores_deep)

In [ ]:
show_scatter_with_threshold(real_data, uncertainty_scores_deep, base_samples_deep, threshold=77, show_removed=False)

In [ ]:

percentiles = [70, 77, 80]

fig, axes = plt.subplots(1, len(percentiles), figsize=(12, 4))

for i, percentile in enumerate(percentiles):
    percentile_score = jnp.percentile(uncertainty_scores_deep, percentile)
    confident_mask = uncertainty_scores_deep <= percentile_score
    filtered_samples = base_samples_deep[confident_mask]
    axes[i].scatter(filtered_samples[:, 0], filtered_samples[:, 1], s=2, alpha=0.5, color='tab:blue')
    axes[i].set_title(f"Filtered Dataset ({percentile}th Percentile)")

for ax in axes:
    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-1.8, 1.8)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.3)
    ax.set_xticks([-1, 0, 1])
    ax.set_yticks([-1, 0, 1])
    
plt.tight_layout()
plt.show()

In [ ]:
# plot of samples from one of the ensemble members
ensemble_samples = ensemble_samples_deep
fig, ax = plt.subplots(figsize=(6, 6))
model_id = 6
ax.scatter(ensemble_samples[model_id, :, 0], ensemble_samples[model_id, :, 1], s=2, alpha=0.5, color='tab:blue')
ax.set_title("Generated Dataset")
ax.set_xlim(-1.8, 1.8)
ax.set_ylim(-1.8, 1.8)
ax.set_aspect('equal', adjustable='box')
ax.grid(True, alpha=0.3)
ax.set_xticks([-1, 0, 1])
ax.set_yticks([-1, 0, 1])
plt.tight_layout()

In [ ]:
# plot of samples from one of the ensemble members
ensemble_samples = ensemble_samples_lora
fig, ax = plt.subplots(figsize=(6, 6))
model_id = 6
target_point = 9101
ax.scatter(real_data[:, 0], real_data[:, 1], s=2, alpha=0.01, color='tab:orange')
# ax.scatter(ensemble_samples[model_id, :, 0], ensemble_samples[model_id, :, 1], s=2, alpha=0.5, color='tab:blue')
for model_id in range(ensemble_samples.shape[0]):
    ax.scatter(ensemble_samples[model_id, target_point, 0], ensemble_samples[model_id, target_point, 1], s=100, alpha=0.9, label=f'Model {model_id}', marker='x')
ax.set_title("Generated Dataset")
ax.set_xlim(-1.8, 1.8)
ax.set_ylim(-1.8, 1.8)
ax.set_aspect('equal', adjustable='box')
ax.grid(True, alpha=0.3)
ax.set_xticks([-1, 0, 1])
ax.set_yticks([-1, 0, 1])
plt.tight_layout()
plt.legend()

In [ ]:
plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({
    'font.size': 14,          # General font size
    'axes.titlesize': 16,     # Title size
    'axes.labelsize': 14,     # X and Y axis label size
    'xtick.labelsize': 12,    # X tick label size
    'ytick.labelsize': 12,    # Y tick label size
    'legend.fontsize': 14# Legend text size
})
show_hallucination_analysis(uncertainty_scores_lora, ensemble_samples_lora[0], threshold=95)

In [ ]:
from generative_uncertainty.scoring import extract_true_gmm_params, evaluate_exact_gmm, gmm_score_percentiles
true_means, true_var = extract_true_gmm_params()

In [ ]:
ensemble_samples_la = runs[0]['ensemble_samples']
uncertainty_scores_la = runs[0]['uncertainty_scores']
df_la_gmm = gmm_score_percentiles(real_data, ensemble_samples_la, uncertainty_scores_la, percentile_step=3)
df_deep_gmm = gmm_score_percentiles(real_data, ensemble_samples_deep, uncertainty_scores_deep, percentile_step=3)
# df_lora_gmm = gmm_score_percentiles(real_data, ensemble_samples_lora, uncertainty_scores_lora, percentile_step=3)
real_data_baseline = evaluate_exact_gmm(real_data, true_means, true_var)

In [ ]:
import numpy as np
import pandas as pd
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Generative Uncertainty Filtering Performance Across Percentiles", fontsize=16)
axes[0].plot(df_deep_gmm.columns, df_deep_gmm.loc['avg_log_likelihood'], marker='o', label='Generated Data (DE-Filter)')
axes[1].plot(df_deep_gmm.columns, df_deep_gmm.loc['mode_kl_divergence'], marker='o', label='Generated Data (DE-Filter)')
axes[2].plot(df_deep_gmm.columns, df_deep_gmm.loc['variance_ratio'], marker='o', label='Generated Data (DE-Filter)')
# axes[2].fill_between(np.asarray(df_deep_mmd.columns), pd.to_numeric(df_deep_mmd.loc['ci_lower']), pd.to_numeric(df_deep_mmd.loc['ci_upper']), color='tab:blue', alpha=0.2, label='95% CI (DE-Filter)')

axes[0].plot(df_la_gmm.columns, df_la_gmm.loc['avg_log_likelihood'], marker='x', label='Generated Data (LA-Filter)')
axes[1].plot(df_la_gmm.columns, df_la_gmm.loc['mode_kl_divergence'], marker='x', label='Generated Data (LA-Filter)')
axes[2].plot(df_la_gmm.columns, df_la_gmm.loc['variance_ratio'], marker='x', label='Generated Data (LA-Filter)')

# axes[0].plot(df_lora_gmm.columns, df_lora_gmm.loc['avg_log_likelihood'], marker='s', label='Generated Data (LoRA-Filter)')
# axes[1].plot(df_lora_gmm.columns, df_lora_gmm.loc['mode_kl_divergence'], marker='s', label='Generated Data (LoRA-Filter)')
# axes[2].plot(df_lora_gmm.columns, df_lora_gmm.loc['variance_ratio'], marker='s', label='Generated Data (LoRA-Filter)')

# axes[2].fill_between(np.asarray(df_la_mmd.columns), pd.to_numeric(df_la_mmd.loc['ci_lower']), pd.to_numeric(df_la_mmd.loc['ci_upper']), color='tab:orange', alpha=0.2, label='95% CI (LA-Filter)')
axes[0].axhline(real_data_baseline['avg_log_likelihood'], color='tab:red', linestyle='--', label='Real Data')
axes[1].axhline(real_data_baseline['mode_kl_divergence'], color='tab:red', linestyle='--', label='Real Data')
axes[2].axhline(real_data_baseline['variance_ratio'], color='tab:red', linestyle='--', label='Real Data')
# axes[2].axhline(ref_mmd['mean'], color='tab:orange', linestyle='--', label='Real Data')
# axes[2].fill_between([70, 100], ref_mmd['ci_lower'], ref_mmd['ci_upper'], color='tab:orange', alpha=0.2, label='95% CI (Real Data)')
axes[0].set_xlabel('Percentile')
axes[0].set_ylabel('Log-Likelihood')
axes[0].set_title(r'Log-Likelihood (Precision) ($\uparrow$)')
axes[1].set_xlabel('Percentile')
axes[1].set_ylabel('KL Divergence')
axes[1].set_title(r'Mode KL Divergence (Recall) ($\downarrow$)')
axes[2].set_xlabel('Percentile')
axes[2].set_ylabel('Variance Ratio')
axes[2].set_title(r'Variance Ratio ($\uparrow$)')
axes[0].grid(True, alpha=0.3)
axes[1].grid(True, alpha=0.3)
axes[2].grid(True, alpha=0.3)
# axes[1].set_xlim(80, 100)
axes[1].set_ylim(-0.001)
axes[0].legend()
axes[1].legend()
axes[2].legend()

In [ ]:
from generative_uncertainty.scoring import mmd_score_percentiles
from generative_uncertainty.scoring import estimator, single_rbf_mmd

In [ ]:
percentiles = [70, 72, 75, 76, 77, 78, 79, 80, 85, 90, 95, 100]
mmd_df_deep = mmd_score_percentiles(real_data, ensemble_samples_deep, uncertainty_scores_deep, gamma=0.2, percentiles=percentiles, num_iterations=30, subsample_size=10000)
mmd_df_la = mmd_score_percentiles(real_data, ensemble_samples_la, uncertainty_scores_la, gamma=0.2, percentiles=percentiles, num_iterations=30, subsample_size=10000)
ref_mmd = estimator(lambda X, Y: single_rbf_mmd(X, Y, gamma=0.2), real_data, real_data, num_iterations=30, subsample_size=10000)

In [ ]:
saved_df = pd.DataFrame(mmd_df_deep)
saved_df.to_csv("mmd_deep_percentiles.csv", index=False)

In [ ]:
import pandas as pd
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(mmd_df_deep.columns, mmd_df_deep.loc['mean'], marker='o', label='Generated Data (DE-Filter)')
ax.fill_between(mmd_df_deep.columns, pd.to_numeric(mmd_df_deep.loc['ci_lower']), pd.to_numeric(mmd_df_deep.loc['ci_upper']), color='tab:blue', alpha=0.2, label='95% CI (DE-Filter)')
ax.axhline(ref_mmd['mean'], color='tab:orange', linestyle='--', label='Real Data')
ax.fill_between(mmd_df_deep.columns, ref_mmd['ci_lower'], ref_mmd['ci_upper'], color='tab:orange', alpha=0.2, label='95% CI (Real Data)')   
ax.set_xlabel('Percentile')
ax.set_ylabel('MMD Score')
ax.set_title('MMD Score Across Percentiles (DE-Filter)')
ax.grid(True, alpha=0.3)
ax.legend()

In [ ]:
import glob
# os.listdir("/dtu/blackhole/13/213811/s243425/gaussian_experiment/samples")
samples = glob.glob("/dtu/blackhole/13/213811/s243425/gaussian_experiment/samples/la_ensemble_*subsetlast*.npy")
# samples = glob.glob("/dtu/blackhole/13/213811/s243425/gaussian_experiment/samples/oft_*.npy")
# samples.extend(glob.glob("/dtu/blackhole/13/213811/s243425/gaussian_experiment/samples/lora_*.npy"))
for i, sample in enumerate(samples):
    print(f"{i}: {sample}")

from generative_uncertainty.utils import get_clean_label

In [ ]:
def get_runs(runs_selected, samples):
    runs = []
    for run in runs_selected:
        ensemble_samples = jnp.load(samples[run])
        label = get_clean_label(samples[run])
        uncertainty_scores = get_uncertainty_scores(ensemble_samples)
        runs.append({
            "label": label,
            "ensemble_samples": ensemble_samples,
            "uncertainty_scores": uncertainty_scores
        })
    return runs

In [ ]:
# runs_selected_prior_only = [2, 3, 4, 5, 6, 7, 8, 9, 10, 0]
# runs_selected = [3, 12, 16]
# runs_selected = [6, 19, 22]
# runs_selected = [0,1]
runs_selected = [5,7,9,11,13,0] # la prior ablation
runs_selected = [13,0,3,4,16] # temperature + prior 1,2 bad
runs = get_runs(runs_selected, samples)

In [ ]:
selected_run = 0
show_scatter_with_threshold(real_data, runs[selected_run]["uncertainty_scores"], runs[selected_run]["ensemble_samples"][0], threshold=80, show_removed=False)

In [ ]:
selected_run = -1
show_hallucination_analysis(runs[selected_run]["uncertainty_scores"], runs[selected_run]["ensemble_samples"][0], threshold=99)

In [ ]:
# plot of samples from one of the ensemble members
ensemble_samples = runs[0]['ensemble_samples']
print(runs[0]['label'])
fig, ax = plt.subplots(figsize=(6, 6))
model_id = 2
ax.scatter(ensemble_samples[model_id, :, 0], ensemble_samples[model_id, :, 1], s=2, alpha=0.5, color='tab:blue')
print(ensemble_samples[model_id, :, 0].shape)
ax.set_title("Generated Dataset")
ax.set_xlim(-1.8, 1.8)
ax.set_ylim(-1.8, 1.8)
ax.set_aspect('equal', adjustable='box')
ax.grid(True, alpha=0.3)
ax.set_xticks([-1, 0, 1])
ax.set_yticks([-1, 0, 1])
plt.tight_layout()

In [ ]:
print(colors)

In [ ]:
selected_colors = [colors['lightorange'], colors['lightblue'], colors['green'], colors['yellow'], colors['blue'], colors['black']]
linestyles = ['--', '-.', '--', '-.', '--', '-.']

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from generative_uncertainty.scoring import uncertainty_alignment_aurc

rejection_rates = np.linspace(0.0, 0.95, 96)


alignment = None

fig, ax = plt.subplots(figsize=(12, 6))

for run in runs:
    alignment = uncertainty_alignment_aurc(
        reference_uncertainty=uncertainty_scores_deep,
        approx_uncertainty=run["uncertainty_scores"],
        rejection_rates=rejection_rates,
    )

    run_split = run['label'].split(',')
    run_label = run_split[0]
    if "Temp" in run_split[1]:
        run_label += " " + run_split[1]
    else:
        run_label += " Temp: 1.0"
    ax.plot(
        alignment["candidate"]["rejection_rates"],
        alignment["candidate"]["risks"],
        # label=f"{run['label'].split(',')[0]} (AURC={alignment['candidate']['aurc']:.6f} )",
        # label=f"{run['label'].split(',')[0]}",
        label=f"{run_label}",
        linewidth=4,
        color=selected_colors[runs.index(run)],
        linestyle=linestyles[runs.index(run)]
    )

ax.plot(
    alignment["oracle"]["rejection_rates"],
    alignment["oracle"]["risks"],
    # label=f"Deep ranking oracle (AURC={alignment['oracle']['aurc']:.6f})",
    label=f"Deep ranking oracle",
    linewidth=3,
    linestyle="--",
    color=colors['orange']
)
ax.plot(
    alignment["random"]["rejection_rates"],
    alignment["random"]["risks"],
    label=f"Random ranking",
    linewidth=3,
    color=colors['blue'],
    linestyle=":",
)
ax.set_xlabel("Rejection Rate")
ax.set_ylabel("Mean Deep-Uncertainty of Kept Samples")
ax.set_title("Ablation of LLLA Priors vs Deep-Ensemble Uncertainty Baseline")
ax.grid(True, alpha=0.3)
ax.legend(frameon=True, fancybox=True, shadow=True)
plt.tight_layout()
plt.show()

aurc_gap_to_oracle = alignment["candidate"]["aurc"] - alignment["oracle"]["aurc"]
print(f"Deep-oracle AURC: {alignment['oracle']['aurc']:.6f}")
print(f"LA AURC:        {alignment['candidate']['aurc']:.6f}")
print(f"AURC gap to oracle: {aurc_gap_to_oracle:.6f} (closer to 0 is better)")
print(f"Pearson r:  {alignment['metrics']['pearson_r']:.6f}")
print(f"Spearman r: {alignment['metrics']['spearman_r']:.6f}")
print(f"MAE (norm): {alignment['metrics']['mae_normalized']:.6f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from generative_uncertainty.scoring import uncertainty_alignment_aurc

rejection_rates = np.linspace(0.0, 0.95, 96)


alignment = None


fig, ax = plt.subplots(figsize=(9, 5))

alignment = uncertainty_alignment_aurc(
    reference_uncertainty=uncertainty_scores_deep,
    approx_uncertainty=uncertainty_scores_lora,
    rejection_rates=rejection_rates,
)

ax.plot(
    alignment["candidate"]["rejection_rates"],
    alignment["candidate"]["risks"],
    label=f"(AURC={alignment['candidate']['aurc']:.6f} )",
    linewidth=2,
)

ax.plot(
    alignment["oracle"]["rejection_rates"],
    alignment["oracle"]["risks"],
    label=f"Deep ranking oracle (AURC={alignment['oracle']['aurc']:.6f})",
    linewidth=2,
    linestyle="--",
)
ax.plot(
    alignment["random"]["rejection_rates"],
    alignment["random"]["risks"],
    label=f"Random ranking (AURC={alignment['random']['aurc']:.6f})",
    linewidth=2,
    color="tab:gray",
    linestyle=":",
)
ax.set_xlabel("Rejection Rate")
ax.set_ylabel("Mean Deep-Uncertainty of Kept Samples")
ax.set_title("LA Uncertainty vs Deep-Ensemble Baseline")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

aurc_gap_to_oracle = alignment["candidate"]["aurc"] - alignment["oracle"]["aurc"]
print(f"Deep-oracle AURC: {alignment['oracle']['aurc']:.6f}")
print(f"LA AURC:        {alignment['candidate']['aurc']:.6f}")
print(f"AURC gap to oracle: {aurc_gap_to_oracle:.6f} (closer to 0 is better)")
print(f"Pearson r:  {alignment['metrics']['pearson_r']:.6f}")
print(f"Spearman r: {alignment['metrics']['spearman_r']:.6f}")
print(f"MAE (norm): {alignment['metrics']['mae_normalized']:.6f}")

## TP-removal metric across runs

This section defines a run-level metric for how well uncertainty removes **true hallucinations**.

- Ground-truth hallucination: sample farther than `sigma_multiplier * std_dev` from its closest true GMM mode.
- For each rejection rate $r$, remove top-$r$ uncertain samples.
- Compute hallucination recall: $\mathrm{TPR}(r)=\frac{TP}{TP+FN}$.
- Summarize with area under the TPR-vs-rejection curve, then normalize between random and oracle:

$$
\mathrm{nAUTC} = \frac{\mathrm{AUC}_{\text{model}} - \mathrm{AUC}_{\text{random}}}{\mathrm{AUC}_{\text{oracle}} - \mathrm{AUC}_{\text{random}} + \varepsilon}
$$

Interpretation: `0` is random ranking, `1` is oracle ranking.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from generative_uncertainty.scoring import extract_true_gmm_params


def compute_tp_removal_curve(
    uncertainty_scores,
    base_samples,
    rejection_rates,
    sigma_multiplier=6.0,
    eps=1e-12,
):
    """
    Compute hallucination-recall vs rejection-rate and a normalized AUC score.

    Returns
    -------
    curve_df : pd.DataFrame
        Columns: rejection_rate, tp_recall, tp, fp, tn, fn, removed_rate.
    summary : dict
        Includes auc_model, auc_random, auc_oracle, normalized_auct, prevalence.
    """
    scores = np.asarray(uncertainty_scores)
    samples = np.asarray(base_samples)

    true_means, true_var = extract_true_gmm_params()
    true_means = np.asarray(true_means)
    std_dev = float(np.sqrt(np.asarray(true_var)))
    dist_threshold = sigma_multiplier * std_dev

    diffs = samples[:, None, :] - true_means[None, :, :]
    distances = np.linalg.norm(diffs, axis=-1)
    min_distances = np.min(distances, axis=1)
    is_true_hallucination = min_distances > dist_threshold

    rejection_rates = np.asarray(rejection_rates, dtype=float)
    rows = []

    for rr in rejection_rates:
        rr = float(np.clip(rr, 0.0, 1.0))
        percentile = (1.0 - rr) * 100.0
        cutoff = np.percentile(scores, percentile)
        is_pred_hallucination = scores > cutoff

        tp = int(np.sum(is_true_hallucination & is_pred_hallucination))
        fp = int(np.sum((~is_true_hallucination) & is_pred_hallucination))
        tn = int(np.sum((~is_true_hallucination) & (~is_pred_hallucination)))
        fn = int(np.sum(is_true_hallucination & (~is_pred_hallucination)))

        tp_recall = tp / (tp + fn + eps)
        removed_rate = np.mean(is_pred_hallucination)

        rows.append(
            {
                "rejection_rate": rr,
                "tp_recall": tp_recall,
                "tp": tp,
                "fp": fp,
                "tn": tn,
                "fn": fn,
                "removed_rate": removed_rate,
            }
        )

    curve_df = pd.DataFrame(rows).sort_values("rejection_rate").reset_index(drop=True)

    x = curve_df["rejection_rate"].to_numpy()
    y = curve_df["tp_recall"].to_numpy()
    prevalence = float(np.mean(is_true_hallucination))

    auc_model = float(np.trapezoid(y, x))
    auc_random = float(np.trapezoid(x, x))

    if prevalence > eps:
        oracle_y = np.minimum(1.0, x / prevalence)
    else:
        oracle_y = np.ones_like(x)
    auc_oracle = float(np.trapezoid(oracle_y, x))

    normalized_auct = (auc_model - auc_random) / (auc_oracle - auc_random + eps)

    summary = {
        "auc_model": auc_model,
        "auc_random": auc_random,
        "auc_oracle": auc_oracle,
        "normalized_auct": normalized_auct,
        "prevalence": prevalence,
        "sigma_multiplier": sigma_multiplier,
    }

    return curve_df, summary


rejection_rates = np.linspace(0.0, 0.95, 96)
sigma_multiplier = 5.0

candidates = {
    "Deep Ensemble": {
        "uncertainty": uncertainty_scores_deep,
        "base_samples": base_samples_deep,
        "color": colors['orange'],
        "linestyle": '--'
    },
    # "LoRA Ensemble": {
    #     "uncertainty": uncertainty_scores_lora,
    #     "base_samples": base_samples_lora,
    # },
}

if "runs" in globals() and runs is not None:
    for i, run in enumerate(runs):
        run_label = run.get("label", f"LA run {i}")
        candidates[f"LA: {run_label}"] = {
            "uncertainty": run["uncertainty_scores"],
            "base_samples": run["ensemble_samples"][0],
            "color": selected_colors[i],
            "linestyle": linestyles[i]
        }

all_curves = {}
all_summaries = []

for name, payload in candidates.items():
    curve_df, summary = compute_tp_removal_curve(
        uncertainty_scores=payload["uncertainty"],
        base_samples=payload["base_samples"],
        rejection_rates=rejection_rates,
        sigma_multiplier=sigma_multiplier,
    )
    all_curves[name] = curve_df
    all_summaries.append(
        {
            # "run": name,
            "run": name,
            "normalized_auct": summary["normalized_auct"],
            "auc_model": summary["auc_model"],
            "auc_random": summary["auc_random"],
            "auc_oracle": summary["auc_oracle"],
            "hallucination_prevalence": summary["prevalence"],
            "color": payload['color'],
            "linestyle": payload['linestyle']
        }
    )

summary_df = pd.DataFrame(all_summaries).sort_values("normalized_auct", ascending=False)
display(summary_df)

fig, axes = plt.subplots(1, 1, figsize=(8, 5))

for name, curve_df in all_curves.items():
    score = summary_df.loc[summary_df["run"] == name, "normalized_auct"].iloc[0]
    # shortened_name = name.split(",")[0] if "LA" in name else name
    if "LA" in name:
        run_split = name.split(',')
        run_label = run_split[0]
        if "Temp" in run_split[1]:
            run_label += " " + run_split[1]
        else:
            run_label += " Temp: 1.0"
    else:
        run_label = name
    axes.plot(
        curve_df["rejection_rate"],
        curve_df["tp_recall"],
        linewidth=3,
        # label=f"{name} (nAUTC={score:.3f})",
        label=f"{run_label}",
        color=summary_df.loc[summary_df["run"] == name, "color"].iloc[0],
        linestyle=summary_df.loc[summary_df["run"] == name, "linestyle"].iloc[0]
    )

axes.plot(rejection_rates, rejection_rates, "k--", alpha=0.6, label="Random baseline")
axes.set_xlabel("Rejection rate")
axes.set_ylabel("True-hallucination removal ratio")
axes.set_title("TP removal curves")
axes.grid(True, alpha=0.3)
axes.legend(fancybox=True, frameon=True, shadow=True)

# plot_df = summary_df.copy()
# plot_df = plot_df.iloc[::-1]
# axes[1].barh(plot_df["run"], plot_df["normalized_auct"], color="tab:blue", alpha=0.8)
# axes[1].axvline(0.0, linestyle="--", color="k", alpha=0.6)
# axes[1].set_xlabel("Normalized AUTC (0=random, 1=oracle)")
# axes[1].set_title("Run ranking by TP-removal quality")
# axes[1].grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.show()